## Streaming Agent Responses

`agent.invoke()` waits until the entire run is complete before returning.
For a production chatbot or CLI tool, users see nothing until all reasoning
and tool calls finish — which can take seconds.

`agent.stream()` yields chunks **as each step completes**, giving real-time feedback.

**Topics covered:**
* `stream_mode="values"` — full state snapshot after every step
* `stream_mode="updates"` — only what changed per step (leaner output)
* `stream_mode="messages"` — token-by-token LLM output
* Streaming with tools
* Streaming with memory (thread_id)

In [1]:
%run langchain_common.py

C:\Users\akhawaja\git\cs4603\wk3_langchain_agents\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


## Setup — a simple tool agent

We will use a calculator agent so we can see tool calls interleaved in the stream.

In [2]:
from typing import Union

Number = Union[int, float]

@tool
def add(a: Number, b: Number) -> float:
    """Add two numbers."""
    return float(a + b)

@tool
def multiply(a: Number, b: Number) -> float:
    """Multiply two numbers."""
    return float(a * b)

agent = create_agent(llm_noreason, [add, multiply])

## 1. `invoke` — blocking, full response at the end

In [3]:
result = agent.invoke({"messages": [HumanMessage(content="What is (3 + 4) * 5?")]})
print(result["messages"][-1].content)

The result of (3 + 4) * 5 is 35.


## 2. `stream_mode="values"` — full state after every node

Each chunk is the **complete message list** at that point in the graph.
Useful when you want to see the entire conversation evolve step by step.

- The loop uses `agent.stream(..., stream_mode="values")` to iterate through execution step by step.
- In `"values"` mode, each streamed chunk contains the **entire current state** (not just deltas).
- For each chunk, the code selects the **last message** in the state, which is usually the newest event.
- It then prints:
    - the step number,
    - the message class (for example, AI/tool message types),
    - and a formatted view of the message via `pretty_print()`.

This is why `"values"` mode is great for learning/debugging: every step shows the full conversation snapshot.

In [4]:
print("=== stream_mode='values' ===")

for i, chunk in enumerate(agent.stream(
    {"messages": [HumanMessage(content="What is (3 + 4) * 5?")]},
    stream_mode="values",
)):
    last_msg = chunk["messages"][-1]
    print(f"\n--- Step {i + 1}: {type(last_msg).__name__} ---")
    last_msg.pretty_print()

=== stream_mode='values' ===

--- Step 1: HumanMessage ---
================================ Human Message =================================

What is (3 + 4) * 5?

--- Step 2: AIMessage ---
================================== Ai Message ==================================
Tool Calls:
  add (call_04cc0a09-b916-4ca0-8d84-962babb63cbf)
 Call ID: call_04cc0a09-b916-4ca0-8d84-962babb63cbf
  Args:
    a: 3
    b: 4

--- Step 3: ToolMessage ---
================================= Tool Message =================================
Name: add

7.0

--- Step 4: AIMessage ---
================================== Ai Message ==================================
Tool Calls:
  multiply (call_7bc92b54-e0f3-42ab-a6aa-303f303e15e7)
 Call ID: call_7bc92b54-e0f3-42ab-a6aa-303f303e15e7
  Args:
    a: 7
    b: 5

--- Step 5: ToolMessage ---
================================= Tool Message =================================
Name: multiply

35.0

--- Step 6: AIMessage ---
================================== Ai Message ========

## 3. `stream_mode="updates"` — only what changed per step

Each chunk is a `{node_name: delta}` dict — only the new messages added by that node.
More efficient than `values`; great for logging or displaying incremental progress.

In [5]:
print("=== stream_mode='updates' ===")

for chunk in agent.stream(
    {"messages": [HumanMessage(content="What is (3 + 4) * 5?")]},
    stream_mode="updates",
):
    for node_name, delta in chunk.items():
        print(f"\n[Node: {node_name}]")
        for msg in delta.get("messages", []):
            msg.pretty_print()

=== stream_mode='updates' ===

[Node: model]
================================== Ai Message ==================================
Tool Calls:
  add (call_fd3ccb94-b2e9-4c90-b86f-9f55b3b26b00)
 Call ID: call_fd3ccb94-b2e9-4c90-b86f-9f55b3b26b00
  Args:
    a: 3
    b: 4

[Node: tools]
================================= Tool Message =================================
Name: add

7.0

[Node: model]
================================== Ai Message ==================================
Tool Calls:
  multiply (call_e692442d-90c6-4c03-9deb-b0deb6f93b86)
 Call ID: call_e692442d-90c6-4c03-9deb-b0deb6f93b86
  Args:
    a: 7
    b: 5

[Node: tools]
================================= Tool Message =================================
Name: multiply

35.0

[Node: model]
================================== Ai Message ==================================

The result of (3 + 4) * 5 is 35.


## 4. `stream_mode="messages"` — token-by-token LLM output

This mode streams individual message chunks (tokens) from the LLM as they are generated.
Each chunk is a `(message_chunk, metadata)` tuple.
Tool messages are yielded whole (not tokenized).

In [6]:
from langchain_core.messages import AIMessageChunk

print("=== stream_mode='messages' — token streaming ===")
print()

for msg_chunk, metadata in agent.stream(
    {"messages": [HumanMessage(content="What is (3 + 4) * 5? Explain each step.")]},
    stream_mode="messages",
):
    # AIMessageChunk tokens arrive one at a time — print without newline for live feel
    if isinstance(msg_chunk, AIMessageChunk) and msg_chunk.content:
        print(msg_chunk.content, end="", flush=True)

print()  # final newline

=== stream_mode='messages' — token streaming ===

To solve the expression $(3 + 4) \times 5$, we follow the order of operations (PEMDAS/BODMAS), which dictates that we perform operations inside parentheses first.

**Step 1: Solve the operation inside the parentheses.**
$$3 + 4 = 7$$

**Step 2: Multiply the result by 5.**
$$7 \times 5 = 35$$

**Final Answer:**
The result is **35**.


## 5. Streaming with memory (thread_id)

Streaming works exactly the same when the agent has a checkpointer.
Pass the `config` dict alongside the stream call.

In [7]:
from langgraph.checkpoint.memory import InMemorySaver

mem_agent = create_agent(llm_noreason, [add, multiply], checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "stream-demo"}}

In [8]:
# Turn 1 — stream the first message
print("Turn 1:")
for msg_chunk, _ in mem_agent.stream(
    {"messages": [HumanMessage(content="My favourite number is 7.")]},
    config,
    stream_mode="messages",
):
    if isinstance(msg_chunk, AIMessageChunk) and msg_chunk.content:
        print(msg_chunk.content, end="", flush=True)
print()

# Turn 2 — agent remembers the favourite number
print("\nTurn 2:")
for msg_chunk, _ in mem_agent.stream(
    {"messages": [HumanMessage(content="What is my favourite number multiplied by 6?")]},
    config,
    stream_mode="messages",
):
    if isinstance(msg_chunk, AIMessageChunk) and msg_chunk.content:
        print(msg_chunk.content, end="", flush=True)
print()

Turn 1:
That's a great number! It's often considered lucky and has some interesting mathematical properties. Is there anything specific you'd like to do with it?

Turn 2:
Your favourite number, 7, multiplied by 6 is 42.


## 6. Utility helper — `print_stream`

A reusable helper that pretty-prints streamed output. This is the pattern
you would use in a CLI chatbot or Jupyter widget.

In [9]:
def print_stream(agent, question: str, config: dict | None = None):
    """Stream an agent response and print tokens as they arrive."""
    kwargs = {"stream_mode": "messages"}
    if config:
        kwargs["config"] = config

    for msg_chunk, _ in agent.stream({"messages": [HumanMessage(content=question)]}, **kwargs):
        if isinstance(msg_chunk, AIMessageChunk) and msg_chunk.content:
            print(msg_chunk.content, end="", flush=True)
    print()


print_stream(agent, "What is 12 + 99, then multiply by 3?")

The result is 333.
